# Credit Card Fraud Detection

**Dataset:** [Kaggle Credit Card Fraud Dataset](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud) — 284,807 transactions from European cardholders (Sept 2013), with only 492 fraud cases (~0.17%).

The main challenge here is that this is a heavily imbalanced classification problem. A model that just predicts "not fraud" every time would get 99.83% accuracy, which is obviously useless. So we need to focus on precision-recall rather than just accuracy.

Features V1–V28 are PCA-transformed (anonymized). Only `Time`, `Amount`, and `Class` are in their original form.

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             precision_recall_curve, average_precision_score,
                             roc_curve, roc_auc_score, f1_score,
                             precision_score, recall_score, accuracy_score)
from imblearn.over_sampling import SMOTE

# gets rid of annoying warnings cluttering the output
import warnings
warnings.filterwarnings('ignore')

# make plots look nicer
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

## Loading the Data

V1-V28 are PCA components (anonymized), `Time` is seconds since first transaction, `Amount` is the transaction value, and `Class` is the target (0 = legit, 1 = fraud).

In [ ]:
df = pd.read_csv('creditcard.csv')
print(df.shape)
df.head()

In [ ]:
df.tail()

In [ ]:
# column names
df.columns.tolist()

In [ ]:
# data types + memory
df.info()

In [ ]:
# any nulls?
df.isnull().sum().sum()

## Basic Stats & Class Distribution

Let's look at the descriptive stats and see how bad the class imbalance really is.

In [ ]:
df[['Time', 'Amount', 'Class']].describe()

In [ ]:
# transposed so its easier to read
df.describe().T

The fraud ratio is basically fraud_count / total * 100. If it's super low (which it is), then accuracy is a bad metric since a "predict all legit" model would score 99.8%+.

In [ ]:
# class distribution
print(df['Class'].value_counts())
print()
print(df['Class'].value_counts(normalize=True) * 100)

# how bad is it
fraud_count = df['Class'].sum()
total = len(df)
print(f"\nFraud ratio: {fraud_count}/{total} = {fraud_count/total*100:.4f}%")
print(f"Roughly 1 fraud per {(total - fraud_count) // fraud_count} legit transactions")

In [ ]:
# visualize the imbalance
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# bar chart
sns.countplot(x='Class', data=df, ax=axes[0])
axes[0].set_title('Transaction Count by Class')
axes[0].set_xticklabels(['Legitimate', 'Fraud'])

# put count labels on bars
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height()):,}',
                     (p.get_x() + p.get_width()/2., p.get_height()),
                     ha='center', va='bottom')

# pie chart
axes[1].pie(df['Class'].value_counts(), labels=['Legitimate', 'Fraud'],
            autopct='%1.3f%%', startangle=90, explode=[0, 0.1])
axes[1].set_title('Class Proportion')

plt.tight_layout()
plt.show()

## Preprocessing - Scaling

V1-V28 are already standardized from PCA, but `Amount` and `Time` are on totally different scales. We need to fix that because it messes with distance-based methods (like SMOTE) and gradient-based models.

StandardScaler just does `z = (x - mean) / std` for each feature.

In [ ]:
scaler = StandardScaler()

# scale amount and time to match PCA features
df['Amount_scaled'] = scaler.fit_transform(df[['Amount']])
df['Time_scaled'] = scaler.fit_transform(df[['Time']])

# drop originals since we have scaled versions now
df.drop(['Amount', 'Time'], axis=1, inplace=True)
df.head()

In [ ]:
# duplicates?
dupes = df.duplicated().sum()
print(f"Duplicates: {dupes}")

if dupes > 0:
    df.drop_duplicates(inplace=True)
    print(f"After dropping: {len(df)} rows")

In [ ]:
print(df.shape)
df.head()

## EDA

Let's see how fraud and legit transactions differ in terms of amount, time, and which PCA features separate them.

In [ ]:
# separate fraud and legit for plots
fraud = df[df['Class'] == 1]
legit = df[df['Class'] == 0]
print(f"Legit: {len(legit)}, Fraud: {len(fraud)}")

In [ ]:
# amount distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.kdeplot(legit['Amount_scaled'], label='Legit', fill=True, alpha=0.3, ax=axes[0])
sns.kdeplot(fraud['Amount_scaled'], label='Fraud', fill=True, alpha=0.3, ax=axes[0])
axes[0].set_title('Amount Distribution')
axes[0].legend()

# violin plot
violin_data = df[['Amount_scaled', 'Class']].copy()
violin_data['Class'] = violin_data['Class'].map({0: 'Legit', 1: 'Fraud'})
sns.violinplot(x='Class', y='Amount_scaled', data=violin_data, ax=axes[1])
axes[1].set_title('Amount Violin Plot')

plt.tight_layout()
plt.show()

Checking if fraud transactions happen at specific times:

In [ ]:
# time distribution - do frauds happen at certain times?
plt.figure(figsize=(12, 4))
sns.kdeplot(legit['Time_scaled'], label='Legit', fill=True, alpha=0.3)
sns.kdeplot(fraud['Time_scaled'], label='Fraud', fill=True, alpha=0.3)
plt.title('Time Distribution')
plt.legend()
plt.show()

Looking at some PCA features - picked ones that seemed to have different distributions for fraud vs legit:

In [ ]:
# some PCA features that seem to separate the classes well
features_to_check = ['V1', 'V2', 'V4', 'V14', 'V17']

fig, axes = plt.subplots(1, len(features_to_check), figsize=(20, 4))
for i, feat in enumerate(features_to_check):
    sns.kdeplot(legit[feat], label='Legit', alpha=0.5, ax=axes[i])
    sns.kdeplot(fraud[feat], label='Fraud', alpha=0.5, ax=axes[i])
    axes[i].set_title(feat)
    if i == len(features_to_check) - 1:
        axes[i].legend()

plt.suptitle('PCA Feature Distributions', y=1.02)
plt.tight_layout()
plt.show()

## Correlation Analysis

Since the V features are PCA components they're already uncorrelated with each other, so the main thing to look at is how each feature correlates with the target (`Class`).

In [ ]:
# how much each feature correlates with fraud
corr_with_class = df.corr()['Class'].drop('Class').sort_values()

plt.figure(figsize=(10, 8))
colors = ['red' if x < 0 else 'blue' for x in corr_with_class]
corr_with_class.plot(kind='barh', color=colors)
plt.title('Feature Correlation with Fraud')
plt.xlabel('Correlation')
plt.tight_layout()
plt.show()

# top correlated features
print("Top positive:")
print(corr_with_class.tail(5))
print("\nTop negative:")
print(corr_with_class.head(5))

In [ ]:
# heatmap of top features
top_feats = list(corr_with_class.head(5).index) + list(corr_with_class.tail(5).index) + ['Class']

plt.figure(figsize=(10, 8))
sns.heatmap(df[top_feats].corr(), annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True)
plt.title('Correlation Heatmap - Top Features')
plt.tight_layout()
plt.show()

## Train-Test Split

Important: splitting **before** SMOTE. If you oversample first then split, synthetic samples could leak into the test set and give inflated results.

In [ ]:
X = df.drop('Class', axis=1)
y = df['Class']

# stratify keeps the same fraud % in train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                    random_state=42, stratify=y)

print(f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")
print(f"Train fraud: {y_train.sum()} ({y_train.mean()*100:.3f}%)")
print(f"Test fraud:  {y_test.sum()} ({y_test.mean()*100:.3f}%)")

## Handling Imbalance with SMOTE

SMOTE generates synthetic minority samples by interpolating between existing fraud cases and their nearest neighbors:

$$x_{new} = x_i + \lambda \cdot (x_{nn} - x_i), \quad \lambda \in [0, 1]$$

This is better than just duplicating fraud rows because it creates new realistic-ish samples. Only applying this to training data obviously.

In [ ]:
# SMOTE on training data only
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print(f"Before: {y_train.value_counts().to_dict()}")
print(f"After:  {pd.Series(y_train_smote).value_counts().to_dict()}")
print(f"Total went from {len(y_train)} to {len(y_train_smote)}")

In [ ]:
# before vs after
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(x=y_train, ax=axes[0])
axes[0].set_title('Before SMOTE')
axes[0].set_xticklabels(['Legit', 'Fraud'])

sns.countplot(x=y_train_smote, ax=axes[1])
axes[1].set_title('After SMOTE')
axes[1].set_xticklabels(['Legit', 'Fraud'])

plt.tight_layout()
plt.show()

## Model Training

Training 3 models to compare:
- **Logistic Regression** — simple baseline, easy to interpret
- **Random Forest** — ensemble of decision trees, handles non-linear patterns
- **XGBoost** — gradient boosting, usually does well on tabular data

### Logistic Regression

In [ ]:
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_smote, y_train_smote)

lr_pred = lr_model.predict(X_test)
lr_proba = lr_model.predict_proba(X_test)[:, 1]  # prob of being fraud

print(f"Acc: {accuracy_score(y_test, lr_pred):.4f}")
print(f"Recall: {recall_score(y_test, lr_pred):.4f}")
print(f"F1: {f1_score(y_test, lr_pred):.4f}")

### Random Forest

Also setting `class_weight='balanced'` which gives more weight to the minority class during training.

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10,
                                  class_weight='balanced',  # helps with imbalance
                                  random_state=42, n_jobs=-1)
rf_model.fit(X_train_smote, y_train_smote)

rf_pred = rf_model.predict(X_test)
rf_proba = rf_model.predict_proba(X_test)[:, 1]

print(f"Acc: {accuracy_score(y_test, rf_pred):.4f}")
print(f"Recall: {recall_score(y_test, rf_pred):.4f}")
print(f"F1: {f1_score(y_test, rf_pred):.4f}")

### XGBoost

XGBoost has a `scale_pos_weight` param that can handle imbalance, but since we already applied SMOTE we set it to 1 here. Calculating it anyway to see the ratio.

In [ ]:
# imbalance ratio (not using it since we have SMOTE but good to know)
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"Imbalance ratio: {scale_pos_weight:.1f}")

xgb_model = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.05,
                           subsample=0.8, colsample_bytree=0.8,
                           scale_pos_weight=1,  # already using SMOTE so set to 1
                           eval_metric='logloss',
                           random_state=42, n_jobs=-1)
xgb_model.fit(X_train_smote, y_train_smote)

xgb_pred = xgb_model.predict(X_test)
xgb_proba = xgb_model.predict_proba(X_test)[:, 1]

print(f"\nAcc: {accuracy_score(y_test, xgb_pred):.4f}")
print(f"Recall: {recall_score(y_test, xgb_pred):.4f}")
print(f"F1: {f1_score(y_test, xgb_pred):.4f}")

## Evaluation

### Precision-Recall Curves

For imbalanced problems, PR curves are way more informative than ROC. The AUPRC (area under PR curve) is really the main metric we should look at here.

- **Precision** = TP / (TP + FP) — of the ones we flagged, how many are actually fraud?
- **Recall** = TP / (TP + FN) — of all actual frauds, how many did we catch?

In [ ]:
models = {
    'Logistic Regression': lr_proba,
    'Random Forest': rf_proba,
    'XGBoost': xgb_proba
}

plt.figure(figsize=(10, 7))
for name, proba in models.items():
    prec, rec, _ = precision_recall_curve(y_test, proba)
    ap = average_precision_score(y_test, proba)
    plt.plot(rec, prec, label=f'{name} (AUPRC={ap:.4f})')

plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curves')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

for name, proba in models.items():
    print(f"{name}: AUPRC = {average_precision_score(y_test, proba):.4f}")

### Confusion Matrices

F1 = 2 * (precision * recall) / (precision + recall) — balances both metrics.

In [ ]:
model_preds = {
    'Logistic Regression': lr_pred,
    'Random Forest': rf_pred,
    'XGBoost': xgb_pred
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, pred) in zip(axes, model_preds.items()):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues', ax=ax,
                xticklabels=['Legit', 'Fraud'], yticklabels=['Legit', 'Fraud'])
    ax.set_title(name)
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')

plt.tight_layout()
plt.show()

# read it as: rows = what it actually is, columns = what the model predicted

In [ ]:
# classification reports + breakdown
for name, pred in model_preds.items():
    print(f"\n--- {name} ---")
    print(classification_report(y_test, pred, target_names=['Legit', 'Fraud']))
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()  # unpack the 4 values
    print(f"  TP: {tp}, FN: {fn}, FP: {fp}, TN: {tn}")

### ROC Curves

Including these for completeness but keep in mind ROC can be misleading here — with so many true negatives, even lots of false positives barely move the FPR needle.

In [ ]:
plt.figure(figsize=(10, 7))
for name, proba in models.items():
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc_score = roc_auc_score(y_test, proba)
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc_score:.4f})')

plt.plot([0, 1], [0, 1], 'k--', label='Random')  # diagonal = random guessing baseline
plt.xlabel('FPR')
plt.ylabel('TPR')
plt.title('ROC Curves')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

for name, proba in models.items():
    print(f"{name}: AUC = {roc_auc_score(y_test, proba):.4f}")

## Threshold Tuning

Default is 0.5 but we can find a better threshold that maximizes F1 for the best model.

In [ ]:
# pick best model by AUPRC
best_name = max(models, key=lambda m: average_precision_score(y_test, models[m]))
best_proba = models[best_name]
print(f"Best: {best_name}")

# try different thresholds to find the one that gives best F1
prec_vals, rec_vals, thresholds = precision_recall_curve(y_test, best_proba)
f1_vals = 2 * (prec_vals[:-1] * rec_vals[:-1]) / (prec_vals[:-1] + rec_vals[:-1] + 1e-8)  # +1e-8 to avoid dividing by 0
best_thresh = thresholds[np.argmax(f1_vals)]
print(f"Best threshold: {best_thresh:.4f} (F1={f1_vals.max():.4f})")

# apply it instead of default 0.5
y_tuned = (best_proba >= best_thresh).astype(int)
print(f"\nWith tuned threshold:")
print(classification_report(y_test, y_tuned, target_names=['Legit', 'Fraud']))

## Feature Importance

Which features are actually driving the predictions? This helps verify the model is learning real patterns and not just noise.

In [ ]:
# feature importance - which features matter most to each model
rf_imp = pd.Series(rf_model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
xgb_imp = pd.Series(xgb_model.feature_importances_, index=X_train.columns).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

rf_imp.head(10).plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Random Forest Top 10')
axes[0].invert_yaxis()  # so highest is on top

xgb_imp.head(10).plot(kind='barh', ax=axes[1], color='salmon')
axes[1].set_title('XGBoost Top 10')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## Cross-Validation

Running 5-fold stratified CV to make sure the results aren't just lucky based on one particular split.

In [ ]:
# quick 5-fold CV on best model to check if results hold
if best_name == 'XGBoost':
    cv_model = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.05,
                              scale_pos_weight=scale_pos_weight,
                              eval_metric='logloss', random_state=42, n_jobs=-1)
elif best_name == 'Random Forest':
    cv_model = RandomForestClassifier(n_estimators=100, max_depth=10,
                                      class_weight='balanced', random_state=42, n_jobs=-1)
else:
    cv_model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_validate(cv_model, X, y, cv=cv, scoring='f1', n_jobs=-1)
print(f"5-Fold CV F1 ({best_name}): {cv_scores['test_score'].mean():.4f} +/- {cv_scores['test_score'].std():.4f}")

## Ethical Considerations

A few things worth noting about deploying something like this:

**False positives vs false negatives** — blocking a legitimate transaction is annoying for the customer, but missing a fraud means real financial loss. There's always a tradeoff and you have to decide which is worse based on the context.

**Dataset limitations** — this is only 2 days of European transactions from 2013. Fraud patterns change over time and vary by region, so a model trained on this wouldn't necessarily work well in other settings. Also, we can only label fraud that was *caught* — if some fraud slipped through undetected, it's mislabeled as legitimate in our data.

**Privacy** — the PCA anonymization is good, but in practice any fraud detection system needs to comply with regulations like GDPR. We also can't really check for demographic bias here since the features are anonymized.

## Results Summary

In [ ]:
# summary table
results = []
for name, pred, proba in [('Logistic Regression', lr_pred, lr_proba),
                           ('Random Forest', rf_pred, rf_proba),
                           ('XGBoost', xgb_pred, xgb_proba)]:
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred),
        'Recall': recall_score(y_test, pred),
        'F1': f1_score(y_test, pred),
        'AUPRC': average_precision_score(y_test, proba),
        'ROC-AUC': roc_auc_score(y_test, proba),
        'TP': tp, 'FN': fn, 'FP': fp
    })

results_df = pd.DataFrame(results).set_index('Model')
results_df

In [ ]:
metrics_to_plot = ['Precision', 'Recall', 'F1', 'AUPRC', 'ROC-AUC']
results_df[metrics_to_plot].plot(kind='bar', figsize=(12, 6), edgecolor='black', width=0.7)
plt.title('Model Comparison')
plt.ylabel('Score')
plt.xticks(rotation=0)
plt.ylim(0, 1.05)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## Takeaways

- The extreme class imbalance (0.17% fraud) means accuracy is useless as a metric — AUPRC is much better for this kind of problem
- SMOTE helped balance the training data. Key thing was applying it *after* the train-test split to avoid data leakage
- All three models got high ROC-AUC but the PR curves showed more meaningful differences
- Threshold tuning gives you control over the precision/recall tradeoff depending on what matters more in practice
- For a real system you'd want to retrain regularly since fraud patterns evolve, and have humans review flagged transactions